### Terms Declaration
Here we declare all the terms (predicates and variables) used in the knowledge base.  
They cover users, places (resources and accommodations), transport, and related constraints such as budget, time, and availability.

In [150]:
from pyDatalog import pyDatalog

pyDatalog.clear()

# ------------------------------------------------------------
# DECLARE TERMS
# ------------------------------------------------------------
pyDatalog.create_terms(
    # Predicates
    'user, resource, transport, accommodation, '
    'within_budget, reachable, time_ok, '
    'accommodation_available, can_stay, '
    'recommend,'
    # Variables
    'User, Budget, TimeAvail, Age, UserLoc, '
    'Resource, ResType, ResLoc, Price, Duration, '
    'FromLoc, ToLoc, Mode, TravelCost, TravelTime, '
    'Capacity, AvailableSpots, '
    'TotalCost, TotalTime'
)

([],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [])

### Knowledge Base: Users and Resources
This cell adds facts to the knowledge base.  
The data covers various entity types and was generated with the assistance of AI.

In [151]:
# user(name, budget, available_hours, age, city)
+user('Alice',   500,  8, 75, 'Rome')
+user('Bob',     300,  6, 40, 'Milan')
+user('Charlie', 800, 10, 30, 'Naples')
+user('Diana',   400,  5, 65, 'Rome')
+user('Eve',     150,  4, 22, 'Florence')

# resource(name, type, city, price, visit_duration)
# --- Milan ---
+resource('Duomo di Milano',     'monument',   'Milan',     10, 2)
+resource('La Scala',            'museum',     'Milan',     18, 2)
+resource('Navigli District',    'park',       'Milan',      0, 3)
+resource('Hotel Milano',        'hotel',      'Milan',    120, 0)
+resource('Ristorante Milanese', 'restaurant', 'Milan',     35, 2)

# --- Rome ---
+resource('Colosseum',           'museum',     'Rome',      15, 3)
+resource('Vatican Museums',     'museum',     'Rome',      20, 4)
+resource('Pantheon',            'museum',     'Rome',       5, 1)
+resource('Hotel Roma',          'hotel',      'Rome',     150, 0)

# --- Naples ---
+resource('Pompeii',             'monument',   'Naples',    18, 4)
+resource('Capodimonte Museum',  'museum',     'Naples',    12, 3)
+resource('Hotel Napoli',        'hotel',      'Naples',    90, 0)
+resource('Pizzeria Napoli',     'restaurant', 'Naples',    20, 1)

# --- Florence ---
+resource('Uffizi Gallery',      'museum',     'Florence',  25, 4)
+resource('Hotel Firenze',       'hotel',      'Florence', 110, 0)

# transport(from, to, mode, cost, duration_hours)
# --- within Milan ---
+transport('Milan',    'Milan',    'walk',     0, 0.5)
+transport('Milan',    'Rome',     'train',   40, 2.0)
+transport('Milan',    'Naples',   'train',   50, 3.0)
+transport('Milan',    'Florence', 'train',   25, 1.5)

# --- within Rome ---
+transport('Rome',     'Rome',     'walk',     0, 0.5)
+transport('Rome',     'Milan',    'train',   50, 3.0)
+transport('Rome',     'Naples',   'train',   20, 1.0)
+transport('Rome',     'Florence', 'train',   30, 1.5)

# --- within Naples ---
+transport('Naples',   'Naples',   'walk',     0, 0.5)
+transport('Naples',   'Rome',     'train',   20, 1.0)
+transport('Naples',   'Milan',    'train',   50, 3.0)

# --- within Florence ---
+transport('Florence', 'Florence', 'walk',     0, 0.5)
+transport('Florence', 'Milan',    'train',   25, 1.5)
+transport('Florence', 'Rome',     'train',   30, 1.5)

# accommodation(city, capacity, available_spots)
+accommodation('Milan',      80,  2)
+accommodation('Rome',      100,  5)
+accommodation('Naples',     50,  3)
+accommodation('Florence',   60,  0)   # full

### Expert Rules and Inference
These rules encode expert knowledge for accessible tourism.  
- `within_budget`: checks if the resource price fits the user's budget.  
- `reachable`: verifies there is a transport connection from the user's city to the resource city.  
- `time_ok`: ensures the visit duration is within the user's available time.  
- `accommodation_available`: checks if there are free spots in the accommodation at a location.  
- `can_stay`: combines user, resource, and accommodation availability.  
- `recommend`: final recommendation requiring all previous conditions.  

This demonstrates rule chaining: `recommend` depends on `within_budget`, `reachable`, `time_ok`, and `can_stay`, which in turn depend on other rules and facts.

In [152]:
# The ticket price fits the budget
within_budget(User, Resource) <= (
    user(User, Budget, TimeAvail, Age, UserLoc) &
    resource(Resource, ResType, ResLoc, Price, Duration) &
    (Price <= Budget)
)

# There is a transport mode from the user's city to the resource's city
reachable(User, Resource) <= (
    user(User, Budget, TimeAvail, Age, UserLoc) &
    resource(Resource, ResType, ResLoc, Price, Duration) &
    transport(UserLoc, ResLoc, Mode, TravelCost, TravelTime)
)

# The visit duration fits the available time
time_ok(User, Resource) <= (
    user(User, Budget, TimeAvail, Age, UserLoc) &
    resource(Resource, ResType, ResLoc, Price, Duration) &
    (Duration <= TimeAvail)
)

# A city has accommodation available if there are free spots
accommodation_available(Location) <= (
    accommodation(Location, Capacity, AvailableSpots) &
    (AvailableSpots > 0)
)

# The user can stay near the resource
can_stay(User, Resource) <= (
    user(User, Budget, TimeAvail, Age, UserLoc) &      
    resource(Resource, ResType, ResLoc, Price, Duration) &
    accommodation_available(ResLoc)
)

# Basic recommendation: all 4 conditions are satisfied
recommend(User, Resource) <= (
    within_budget(User, Resource) &
    reachable(User, Resource) &
    time_ok(User, Resource) &
    can_stay(User, Resource)
)

recommend(User,Resource) <= within_budget(User,Res

### Helper Function: `show`
Utility function to display query results in a clean, readable format.  

In [153]:
def show(title, results):
    """Print a list of results in a clean, sorted way."""
    print(f"\n=== {title} ===")
    if not results:
        print("  (no results)")
        return
    for row in results:
        print("  " + " -> ".join(str(x) for x in row))

In [154]:
show("Resources within Bob's budget (300€)",
     within_budget('Bob', Resource).data)

show("Resources reachable from Bob (Milan)",
     reachable('Bob', Resource).data)

show("Resources that fit Charlie's time (10h)",
     time_ok('Charlie', Resource).data)

show("Basic recommendations for Charlie",
     recommend('Charlie', Resource).data)




=== Resources within Bob's budget (300€) ===
  Hotel Firenze
  Uffizi Gallery
  Pizzeria Napoli
  Hotel Napoli
  Capodimonte Museum
  Pompeii
  Hotel Roma
  Pantheon
  Vatican Museums
  Colosseum
  Ristorante Milanese
  Hotel Milano
  Navigli District
  La Scala
  Duomo di Milano

=== Resources reachable from Bob (Milan) ===
  Hotel Firenze
  Uffizi Gallery
  Pizzeria Napoli
  Hotel Napoli
  Capodimonte Museum
  Pompeii
  Hotel Roma
  Pantheon
  Vatican Museums
  Colosseum
  Ristorante Milanese
  Hotel Milano
  Navigli District
  La Scala
  Duomo di Milano

=== Resources that fit Charlie's time (10h) ===
  Hotel Firenze
  Uffizi Gallery
  Pizzeria Napoli
  Hotel Napoli
  Capodimonte Museum
  Pompeii
  Hotel Roma
  Pantheon
  Vatican Museums
  Colosseum
  Ristorante Milanese
  Hotel Milano
  Navigli District
  La Scala
  Duomo di Milano

=== Basic recommendations for Charlie ===
  Pizzeria Napoli
  Hotel Napoli
  Capodimonte Museum
  Pompeii
  Hotel Roma
  Pantheon
  Vatican Museums
  